In [1]:
# python version used: 3.12.12

In [1]:
!pip install -q rouge-score==0.1.2 nltk==3.9.1 bert-score==0.3.13 litellm==1.81.13 transformers==5.0.0 torch==2.10.0

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 17.3 MB/s eta 0:00:00


In [3]:
source_text = """
The Amazon rainforest, often referred to as the "lungs of the Earth," produces
approximately 20 percent of the world's oxygen. Spanning over 5.5 million
square kilometers across nine countries in South America, it is the largest
tropical rainforest on the planet. The Amazon is home to an estimated 10
percent of all species on Earth, including over 40,000 plant species, 1,300
bird species, and 3,000 types of fish. Deforestation, primarily driven by
cattle ranching and soybean farming, poses the greatest threat to this
ecosystem. Between 2001 and 2020, the Amazon lost approximately 10 percent
of its forest cover. Scientists warn that continued deforestation could push
the rainforest past a tipping point, transforming large portions into savanna
and releasing billions of tons of stored carbon dioxide into the atmosphere.
"""

reference_summary = (
    "The Amazon rainforest is the world's largest tropical forest, producing "
    "about 20% of global oxygen and hosting 10% of Earth's species. "
    "Deforestation from agriculture threatens to push it past a tipping point, "
    "potentially converting it to savanna and releasing massive carbon emissions."
)

# Candidate A: a good summary that captures key points in different words
candidate_a = (
    "Spanning 5.5 million square kilometers, the Amazon is Earth's biggest "
    "tropical rainforest and a critical oxygen source. It harbors extraordinary "
    "biodiversity, but agricultural deforestation — mainly cattle and soy — "
    "risks triggering an irreversible shift to savanna, which would unleash "
    "vast carbon stores."
)

# Candidate B: a flawed summary — vague, misses key facts, adds a hallucination
candidate_b = (
    "The Amazon is a large forest in South America. It has many trees and "
    "animals. Some people cut down trees there. The forest was designated a "
    "UNESCO World Heritage Site in 2005."  # <-- hallucinated fact
)

In [4]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_bleu(candidate, reference):
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()
    # Smoothing handles cases where higher-order n-grams have zero overlap
    smoothie = SmoothingFunction().method1
    return sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)

bleu_a = compute_bleu(candidate_a, reference_summary)
bleu_b = compute_bleu(candidate_b, reference_summary)

print(f"BLEU — Candidate A: {bleu_a:.4f}")
print(f"BLEU — Candidate B: {bleu_b:.4f}")

BLEU — Candidate A: 0.0147
BLEU — Candidate B: 0.0123


In [5]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(candidate, reference):
    scores = scorer.score(reference, candidate)
    return {key: round(val.fmeasure, 4) for key, val in scores.items()}

rouge_a = compute_rouge(candidate_a, reference_summary)
rouge_b = compute_rouge(candidate_b, reference_summary)

print(f"ROUGE — Candidate A: {rouge_a}")
print(f"ROUGE — Candidate B: {rouge_b}")

ROUGE — Candidate A: {'rouge1': 0.3908, 'rouge2': 0.0706, 'rougeL': 0.2529}
ROUGE — Candidate B: {'rouge1': 0.2368, 'rouge2': 0.027, 'rougeL': 0.1579}


In [ ]:
from bert_score import score as bert_score

def compute_bertscore(candidates, references):
    P, R, F1 = bert_score(candidates, references, model_type="roberta-large", lang="en", verbose=False)
    return P.tolist(), R.tolist(), F1.tolist()

# Compute scores
precision_scores, recall_scores, f1_scores = compute_bertscore(
    [candidate_a, candidate_b],
    [reference_summary, reference_summary]
)

# Candidate A
print(f"Candidate A — Precision: {precision_scores[0]:.4f}")
print(f"Candidate A — Recall:    {recall_scores[0]:.4f}")
print(f"Candidate A — F1:        {f1_scores[0]:.4f}")

print()

# Candidate B
print(f"Candidate B — Precision: {precision_scores[1]:.4f}")
print(f"Candidate B — Recall:    {recall_scores[1]:.4f}")
print(f"Candidate B — F1:        {f1_scores[1]:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Candidate A — Precision: 0.8948
Candidate A — Recall:    0.9078
Candidate A — F1:        0.9012

Candidate B — Precision: 0.8875
Candidate B — Recall:    0.8474
Candidate B — F1:        0.8670


In [ ]:
import os
os.environ["OPENROUTER_API_KEY"] = "..."

In [8]:
import litellm
import json

client = litellm.LiteLLM()

JUDGE_PROMPT = """You are an expert evaluator assessing the quality of text summaries.

Given a source text, a reference summary (gold standard), and a candidate summary evaluate
the candidate on three criteria.

For each criterion, assign a score of 0 (fail) to 1 (pass) and provide a
brief justification.

Compare against both the source text AND the reference summary.

**Criteria:**
1. **Accuracy** — Are all claims in the candidate summary factually supported by the source?
   Any hallucinated or fabricated information => accuracy=0.
2. **Completeness** — Does the candidate summary capture ALL the key points of the source given the reference?
   Any missing key points => completeness=0.
3. **Conciseness** — Is the candidate summary brief? Or is it too verbose compared to reference summary?
   Significantly extra verbosity in candidate, given the reference => conciseness=0.

### Response format
Respond with ONLY a valid JSON object (no markdown fences, no commentary):
{
  "accuracy":     {"score": <0 or 1>, "justification": "<1-2 sentences>"},
  "completeness": {"score": <0 or 1>, "justification": "<1-2 sentences>"},
  "conciseness":  {"score": <0 or 1>, "justification": "<1-2 sentences>"}
}
"""

DATA_PROMPT = """\
SOURCE TEXT:
{source}

REFERENCE SUMMARY:
{reference}

CANDIDATE SUMMARY:
{candidate}
"""

def llm_judge(source, reference, candidate, max_retries=2):

    for attempt in range(max_retries + 1):
        resp = client.chat.completions.create(
            model="openrouter/openai/gpt-4o-2024-08-06",
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content": DATA_PROMPT.format(
                    source=source, reference=reference, candidate=candidate
                )},
            ],
            temperature=0.00,
        )

        text = resp.choices[0].message.content.strip()

        # Robust JSON extraction: handle markdown fences (if suppose the model adds them)
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        # Hard-validate JSON; retry once if invalid
        try:
            data = json.loads(text)
            # minimal schema check
            for k in ("accuracy", "completeness", "conciseness"):
                assert k in data and "score" in data[k] and "justification" in data[k]
                assert data[k]["score"] in (0, 1)
            return data
        except Exception:
            if attempt == max_retries:
                # return raw for debugging if it keeps failing
                return {"error": "invalid_json", "raw": text}

# overall score
def overall_score(data):
    scores = [data[k]["score"] for k in ("accuracy", "completeness", "conciseness")]
    average_score = sum(scores) / len(scores)
    print(f"\nAverage score: {average_score}")
    if data["accuracy"]["score"] == 1 and average_score >= 0.65:
        return "PASS"
    else:
        return "FAIL"


print("=" * 60)
print("JUDGE EVALUATION — CANDIDATE A")
print("=" * 60)
data1 = llm_judge(source_text, reference_summary, candidate_a)
print(json.dumps(data1, indent=2))
print ("\n Overall:")
print(overall_score(data1))


print("\n" + "=" * 60)
print("JUDGE EVALUATION — CANDIDATE B")
print("=" * 60)
data2 = llm_judge(source_text, reference_summary, candidate_b)
print(json.dumps(data2, indent=2))
print ("\n Overall:")
print(overall_score(data2))

JUDGE EVALUATION — CANDIDATE A
{
  "accuracy": {
    "score": 1,
    "justification": "All claims in the candidate summary are factually supported by the source text, including the size, biodiversity, and deforestation threats."
  },
  "completeness": {
    "score": 1,
    "justification": "The candidate summary captures all key points from the source text, including the size, oxygen production, biodiversity, and deforestation risks."
  },
  "conciseness": {
    "score": 1,
    "justification": "The candidate summary is concise and comparable in length to the reference summary, without unnecessary verbosity."
  }
}

 Overall:

Average score: 1.0
PASS

JUDGE EVALUATION — CANDIDATE B
{
  "accuracy": {
    "score": 0,
    "justification": "The candidate summary inaccurately claims the Amazon was designated a UNESCO World Heritage Site in 2005, which is not supported by the source text."
  },
  "completeness": {
    "score": 0,
    "justification": "The candidate summary misses key points 